# 01 – Data Quality & Governance Check

Purpose:
- Validate structural integrity (schema + constraints)
- Confirm governed staging policies were applied (e.g., Exposure cap)
- Confirm severity join coverage (matched vs unmatched claims)
- Produce auditable QC artifacts and approve dataset for modeling

### Inputs expected
- `data/staging/freq_staged.parquet`
- `data/staging/sev_staged.parquet`

If these staged files (or the staging/join reports) are missing, this notebook will try to rebuild them **from the latest raw snapshots** in `data/raw_snapshots/`.

### Artifacts produced/checked
- `artifacts/reports/freq_validation_report.json`
- `artifacts/reports/sev_validation_report.json`
- `artifacts/reports/staging_report.json` (checked)
- `artifacts/reports/sev_join_report.json` (checked)
- `data/staging/sev_train.parquet`, `data/staging/sev_unmatched_claims.parquet` (checked/created)

In [14]:
import os
import hashlib
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "notebooks").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
STAGING_DIR = DATA_DIR / "staging"
RAW_SNAP_DIR = DATA_DIR / "raw_snapshots"

ART = ROOT / "artifacts" / "reports"
ART.mkdir(parents=True, exist_ok=True)
STAGING_DIR.mkdir(parents=True, exist_ok=True)

FREQ = STAGING_DIR / "freq_staged.parquet"
SEV = STAGING_DIR / "sev_staged.parquet"
SEV_TRAIN = STAGING_DIR / "sev_train.parquet"
SEV_UNMATCHED = STAGING_DIR / "sev_unmatched_claims.parquet"

STAGING_REPORT = ART / "staging_report.json"
SEV_JOIN_REPORT = ART / "sev_join_report.json"


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def latest_snapshot(pattern: str) -> Path | None:
    paths = sorted(RAW_SNAP_DIR.glob(pattern))
    return paths[-1] if paths else None


# If staged inputs are missing but raw snapshots exist, rebuild staging + report.
if (not FREQ.exists()) or (not SEV.exists()) or (not STAGING_REPORT.exists()):
    freq_snap = latest_snapshot("freMTPL2freq__*.parquet")
    sev_snap = latest_snapshot("freMTPL2sev__*.parquet")

    if freq_snap and sev_snap:
        from src.data.staging import stage_freq_and_sev

        stage_freq_and_sev(
            freq_snapshot_path=freq_snap,
            sev_snapshot_path=sev_snap,
            out_dir=STAGING_DIR,
            report_path=STAGING_REPORT,
            exposure_cap=1.0,
        )
    else:
        missing = [str(p.relative_to(ROOT)) for p in [FREQ, SEV] if not p.exists()]
        raise FileNotFoundError(
            "Missing staged inputs: "
            + ", ".join(missing)
            + ".\n\n"
            + "Either generate them via `src.data.ingest` → `src.data.staging`, "
            + "or place raw snapshots in: "
            + str(RAW_SNAP_DIR.relative_to(ROOT))
        )


# If severity training join/report missing, build them.
if (not SEV_TRAIN.exists()) or (not SEV_UNMATCHED.exists()) or (not SEV_JOIN_REPORT.exists()):
    from src.data.joins import build_severity_training_dataset

    build_severity_training_dataset(
        freq_staged_path=FREQ,
        sev_staged_path=SEV,
        out_path=SEV_TRAIN,
        report_path=SEV_JOIN_REPORT,
    )


# Back-compat names used later in notebook
freq_path = FREQ
sev_path = SEV

{
    "repo_root": str(ROOT),
    "freq_sha256": sha256_file(freq_path),
    "sev_sha256": sha256_file(sev_path),
}

{'freq_sha256': 'a88923b42b6c8e2994335d88cb48df0b555e6d2369b3192a57c3d8062fad016c',
 'sev_sha256': '32c42d2aea005845c8c4a30020b8cfbb73d6cfbf7a7ca31a0c3a62d89ec4fcce'}

In [11]:
import pandas as pd

from src.data.validate import validate_dataset, save_report
from src.data.schemas import FREQ_SCHEMA, SEV_SCHEMA


df_freq = pd.read_parquet(FREQ)
df_sev = pd.read_parquet(SEV)

rep_freq = validate_dataset(
    df_freq,
    FREQ_SCHEMA,
    coerce=True,
    constraints_kwargs={"exposure_cap_policy": 1.0},
)
rep_sev = validate_dataset(df_sev, SEV_SCHEMA, coerce=True)

save_report(rep_freq, ART / "freq_validation_report.json")
save_report(rep_sev, ART / "sev_validation_report.json")

rep_freq.ok, rep_sev.ok, len(rep_freq.findings), len(rep_sev.findings)

(True, True, 1, 0)

### Validation Result

Both datasets pass schema validation.

- Frequency findings: 1 (Exposure cap policy warning)
- Severity findings: 0

No structural errors detected.

In [12]:
import json

staging_rep = json.loads(STAGING_REPORT.read_text(encoding="utf-8"))

# Exposure cap policy
freq_policies = staging_rep["policies"][0]["policies"]
cap = [p for p in freq_policies if p.get("policy") == "cap_exposure"][0]
idpol = [p for p in freq_policies if p.get("policy") == "canonicalize_idpol_to_int64"][0]

cap, idpol

({'policy': 'cap_exposure',
  'cap_value': 1.0,
  'rows_capped': 1224,
  'max_before': 2.01,
  'max_after': 1.0},
 {'policy': 'canonicalize_idpol_to_int64',
  'nulls': 0,
  'bad_parse': 0,
  'non_integer_like': 0,
  'dtype_after': 'int64'})

### Exposure Policy Decision

1224 rows had Exposure > 1 (max 2.01).
Given policy-year interpretation of Exposure,
we cap at 1.0 in staging to ensure frequency offset stability.

This decision is documented in staging policy artifacts.

In [13]:
import json

join_rep = json.loads(SEV_JOIN_REPORT.read_text(encoding="utf-8"))
join_rep["diagnostics"]

{'claims_rows_input': 26639,
 'rows_matched': 26444,
 'rows_unmatched': 195,
 'match_rate': 0.9926799054018545,
 'dup_policies_in_freq': 0,
 'missing_unique_idpol': 6,
 'top_missing_idpol_counts': {'2262511': 66,
  '2282134': 36,
  '2227533': 25,
  '2220367': 24,
  '2277846': 23,
  '2286775': 21},
 'merge_validate': 'many_to_one'}

In [15]:
import pandas as pd

freq = pd.read_parquet(freq_path)

{
    "exposure_gt1": int((freq["Exposure"] > 1).sum()),
    "exposure_lt001": int((freq["Exposure"] < 0.01).sum()),
    "exposure_max": float(freq["Exposure"].max()),
}

{'exposure_gt1': 0, 'exposure_lt001': 6877, 'exposure_max': 1.0}

# Data Approved for Modeling

✔ Schema validated  
✔ Exposure policy enforced  
✔ Join coverage acceptable  
✔ No structural inconsistencies  

Proceed to EDA.